In [ ]:
import os
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from pathlib import Path

In [ ]:

dataframes = []
arquivos = os.listdir(local_destino)
arquivos_csv = [arquivo for arquivo in arquivos if arquivo.lower().endswith('.csv') and '-sp' in arquivo.lower()]
print(arquivos_csv)
arquivos_csv.sort()
primeiro_arquivo = True
colunas_esperadas = None

for arquivo_csv in arquivos_csv:
    caminho_arquivo = os.path.join(local_destino, arquivo_csv)

    try:
        if primeiro_arquivo:

            df = pd.read_csv(caminho_arquivo)
            colunas_esperadas = df.columns.tolist()
            primeiro_arquivo = False
        else:

            df = pd.read_csv(caminho_arquivo, header=0)

            if df.columns.tolist() != colunas_esperadas:
                print(f"As colunas do arquivo {arquivo_csv} não correspondem às colunas esperadas.")

                df = df[colunas_esperadas]
    except Exception as e:
        print(f"Erro ao ler o arquivo {arquivo_csv}: {e}")
        continue
    
    dataframes.append(df)

# Concatenar todos os DataFrames
if dataframes:
    df_concatenado = pd.concat(dataframes, ignore_index=True)
    caminho_saida = os.path.join('sp_10min.csv')
    df_concatenado.to_csv(caminho_saida, index=False)
    print(f"Arquivos concatenados com sucesso. Arquivo salvo em: {caminho_saida}")
else:
    print("Nenhum DataFrame para concatenar.")


In [ ]:
#df = pd.read_csv('completo vazao atraso traceroute 14-07-2024 3horas.csv')
df = pd.read_csv('completo vazao atraso traceroute 14-07-2024 30min.csv')
#df = pd.read_csv('completo vazao atraso traceroute 14-07-2024 10min.csv')
#df = pd.read_csv('completo vazao atraso traceroute 14-07-2024 6horas.csv')

In [ ]:
def perform_pca(df, target_col=None, n_components=None):
    df_pca = df.drop(columns=[target_col], errors='ignore') if target_col else df.copy()
    df_pca = df_pca.select_dtypes(include=[float, int])

    scaler = StandardScaler()
    df_scaled = scaler.fit_transform(df_pca)
    pca = PCA(n_components=n_components) 
    pca.fit(df_scaled)
    feature_importance = pd.DataFrame(
        pca.components_, 
        columns=df_pca.columns, 
        index=[f'PC{i+1}' for i in range(pca.n_components_)]
    )

    explained_variance = pca.explained_variance_ratio_

    print("Importância das features para cada componente principal:")
    print(feature_importance)
    
    print("\nVariância explicada por cada componente:")
    for i, var in enumerate(explained_variance, 1):
        print(f'PC{i}: {var:.4f}')
        
    return feature_importance, explained_variance


feature_importance, explained_variance = perform_pca(df, target_col='Timestamp_cubic', n_components=None)

In [ ]:
def prepare_timestamp_feature(df, timestamp_col, method="numeric"):
    df = df.copy()
    
    if method == "numeric":
        # Conversão direta para formato numérico (por exemplo, segundos desde epoch)
        df['timestamp_numeric'] = pd.to_datetime(df[timestamp_col]).astype(int) / 10**9  # em segundos
    elif method == "components":
        # Extração de componentes (ano, mês, dia, hora, minuto)
        df['year'] = pd.to_datetime(df[timestamp_col]).dt.year
        df['month'] = pd.to_datetime(df[timestamp_col]).dt.month
        df['day'] = pd.to_datetime(df[timestamp_col]).dt.day
        df['hour'] = pd.to_datetime(df[timestamp_col]).dt.hour
        df['minute'] = pd.to_datetime(df[timestamp_col]).dt.minute
        print(df)
    else:
        raise ValueError("Método inválido. Use 'numeric' ou 'components'.")
    
    return df.drop(columns=[timestamp_col])

def perform_pca_with_timestamp(df, target_col=None, n_components=None, timestamp_col='timestamp', timestamp_method="numeric"):
    # Preparação do timestamp como feature
    df_pca = prepare_timestamp_feature(df, timestamp_col, method=timestamp_method)
    
    # Remover colunas não numéricas
    df_pca = df_pca.select_dtypes(include=[float, int])

    # Normalização dos dados
    scaler = StandardScaler()
    df_scaled = scaler.fit_transform(df_pca)
    
    # Aplicação do PCA
    pca = PCA(n_components=n_components)  # número de componentes desejado
    pca.fit(df_scaled)
    
    # Coeficientes de importância das features
    feature_importance = pd.DataFrame(
        pca.components_, 
        columns=df_pca.columns, 
        index=[f'PC{i+1}' for i in range(pca.n_components_)]
    )
    
    # Variância explicada por cada componente principal
    explained_variance = pca.explained_variance_ratio_

    print("Importância das features para cada componente principal:")
    print(feature_importance)
    
    print("\nVariância explicada por cada componente:")
    for i, var in enumerate(explained_variance, 1):
        print(f'PC{i}: {var:.4f}')
        
    return feature_importance, explained_variance

feature_importance, explained_variance = perform_pca_with_timestamp(df, target_col='Vazao_bbr', n_components=None, timestamp_col='Timestamp_cubic', timestamp_method="numeric")



In [ ]:
def index_adjustment(*dfs):
    resampled_dfs = []

    for i, df in enumerate(dfs):
        df['timestamp'] = pd.to_datetime(df['Timestamp_cubic'], unit='s')
        df.set_index('timestamp', inplace=True)

        df_resampled = df.resample('6h').agg({
            'Vazao': 'mean',        
            'Vazao_bbr': 'mean',   
            'Hop_count': 'mean',   
            'Atraso(ms)': 'mean',   
            'Bottleneck': 'min'     
        })
        df_resampled.dropna(inplace = True)

        df_resampled.drop(columns = ['Hop_count', 'Atraso(ms)', 'Bottleneck'], inplace = True)
        
        resampled_dfs.append(df_resampled)

    common_timestamps = resampled_dfs[0].index
    for df_resampled in resampled_dfs[1:]:
        common_timestamps = common_timestamps.intersection(df_resampled.index)
    
    #print("Timestamps comuns após o resample:", common_timestamps)
    adjusted_dfs = []
    for i, df_resampled in enumerate(resampled_dfs):
        df_common = df_resampled.reindex(common_timestamps)
        adjusted_dfs.append(df_common)
        #print(f"DataFrame {i+1} após reindexação com timestamps comuns:", df_common)
    
    correlations = {}
    for i in range(len(adjusted_dfs)):
        for j in range(i + 1, len(adjusted_dfs)):
            corr = adjusted_dfs[i].corrwith(adjusted_dfs[j])
            correlations[f'DF{i+1}_DF{j+1}'] = corr
            print(f"Correlação entre DataFrame {i+1} e DataFrame {j+1}:\n{corr}\n")
    

    return adjusted_dfs, correlations

folder_path = Path('../datasets/serie-multivariada/')
dataframes = []
for file_path in folder_path.glob('*.csv'):
    if '-am-m' in file_path.stem:
        print(file_path.stem)
        df = pd.read_csv(file_path)
        dataframes.append(df)
adjusted_dfs, correlations = index_adjustment(*dataframes)


print("DataFrames ajustados:", adjusted_dfs)
print("Correlação entre pares de DataFrames:", correlations)


In [ ]:
def index_adjustment(*dfs):
    resampled_dfs = []

    for i, df in enumerate(dfs):
        df['timestamp'] = pd.to_datetime(df['Timestamp_cubic'], unit='s')
        df.set_index('timestamp', inplace=True)

        df_resampled = df.resample('6h').agg({
            'Vazao': 'mean',        
            'Vazao_bbr': 'mean',   
            'Hop_count': 'mean',   
            'Atraso(ms)': 'mean',   
            'Bottleneck': 'min'     
        })
        df_resampled.dropna(inplace=True)
        #df_resampled.drop(columns=['Hop_count', 'Atraso(ms)', 'Bottleneck'], inplace=True)
        
        resampled_dfs.append(df_resampled)

    # Encontrar timestamps comuns
    common_timestamps = resampled_dfs[0].index
    for df_resampled in resampled_dfs[1:]:
        common_timestamps = common_timestamps.intersection(df_resampled.index)

    adjusted_dfs = []
    for i, df_resampled in enumerate(resampled_dfs):
        df_common = df_resampled.reindex(common_timestamps)
        adjusted_dfs.append(df_common)

    correlations = {}
    for i in range(len(adjusted_dfs)):
        for j in range(i + 1, len(adjusted_dfs)):
            corr = adjusted_dfs[i].corrwith(adjusted_dfs[j])
            correlations[f'DF{i+1}_DF{j+1}'] = corr
    
    return adjusted_dfs, correlations

# Caminho da pasta com arquivos CSV
folder_path = Path('../datasets/serie-multivariada/')
output_file = 'correlations_output.txt'


dataframes = []


for file_path in folder_path.glob('*.csv'):
    link = file_path.name.split('-')[2]+'-'+file_path.name.split('-')[3]
    if '-am-m' in file_path.stem:
        print(f"Carregando arquivo: {file_path.stem}")
        df = pd.read_csv(file_path)
        dataframes.append(df)
adjusted_dfs, correlations = index_adjustment(*dataframes)


with open(output_file, 'w') as f:
    f.write("Correlação entre pares de DataFrames:\n")
    for pair, corr in correlations.items():
        f.write(f"\n{pair}:\n{corr}\n")

print(f"Correlação entre pares de DataFrames foi salva em '{output_file}'")
